# assists term — E[assists] one gameweek ahead

**Render-not-decide.** A re-runnable view of the `assists` term, **not** the record — the frozen numbers live in [predictive-phase2-component-model.md](../../../docs/studies/results/predictive-phase2-component-model.md). All logic is in `model/terms/assists/` (+ the shared Poisson-player base); the notebook only calls it and plots.

Same shape as goals; the primary creative position is **MID**. Population: `minutes > 0`, DGW excluded, GW > 3, conditional on appearance (see `ASSUMPTIONS.md`).

## Setup

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart
from model.terms.assists import AssistsModel, AssistsTerm

mart = load_mart()
term = AssistsTerm(AssistsModel(variant="minimal"))
term.model.pool.minimal, [f.name for f in term.model.pool.candidates]

## Pre-fit assumptions
> Is Poisson justified (material dispersion ~1) and is the effect learnable at this N? See `ASSUMPTIONS.md` §1, §4.

In [ ]:
report = term.model.check_assumptions(AssistsModel.population(mart))
print(f"family_ok={report.family_ok}  detectable={report.detectable}  n_train={report.n_train}")
report.dispersion

## Gate — E[assists] vs the term's own lagged-assists baseline
> Per-term level (spec §5): does the signal-based model out-rank a player's naive assists history, within position? MID is the position to watch.

In [ ]:
gate = term.validate(mart)
display(gate.table)

ax = gate.table.set_index("position")[["baseline", "e_assists"]].plot.bar(figsize=(7, 3.5))
ax.set_ylabel("within-position Spearman"); ax.set_title("assists: model vs own baseline"); plt.tight_layout()

## Diagnose — residuals + ablation
> Post-gate. Residuals: the worst-missed player-GWs. Ablation: drop each feature, re-score — the measured contribution of `xgi_roll3` vs `minutes_roll3`.

In [ ]:
diag = term.diagnose(mart)
display(diag.ablation)
display(diag.residuals.head(10))